# 🎬 ClipForge AI — *PecinoGP Edition*

### Tu Opus Clip privado, ejecutándose en Google Colab sobre tus directos de Google Drive

Convierte un directo completo en **clips verticales 9:16 listos para TikTok / Shorts / Reels**, con:

| Función | Cómo lo hace |
|---|---|
| 🧠 Detección de momentos virales | Whisper + análisis acústico (librosa) + embeddings semánticos + LLM director (Gemini) |
| ⭐ Score de viralidad 0-100 | Fusión de 6 señales + veredicto del LLM |
| ✂️ Duración automática | Elige entre 20 / 30 / 45 / 60 s según el contenido |
| 🗣️ Subtítulos profesionales | Palabra a palabra, karaoke con rebote, blanco + borde negro + resaltado |
| 😂 Emojis contextuales | Solo cuando tienen sentido (LLM o mapa semántico) |
| 🎥 Reencuadre inteligente | MediaPipe: sigue la cara del hablante con paneo suave |
| 🔇 Eliminación de silencios | Comprime pausas largas automáticamente |
| ⚡ Efectos dinámicos | Zoom, punch zoom, shake, flash, motion blur |
| 🎚️ Audio broadcast | De-noise, compresor, EQ, loudnorm -14 LUFS, limitador, música con ducking |
| 📦 Export + metadatos | `clip_001.mp4` + `clip_001.json` (score, tema, resumen, keywords) |

---

## ⚡ Antes de ejecutar (2 minutos)

1. **GPU**: menú `Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU`.
2. **(Recomendado, gratis)**: crea una API key en [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey) y guárdala en los **Secretos de Colab** (icono 🔑 del panel izquierdo) con el nombre `GEMINI_API_KEY`. Con ella, la selección de momentos, títulos y emojis los decide un LLM. Sin ella, el sistema funciona igualmente en modo heurístico.
3. `Entorno de ejecución ▸ Ejecutar todas`.

**Tiempos orientativos (directo de 2 h, GPU T4):** transcripción ≈ 25-40 min con `large-v3` (≈ 10-12 min con `large-v3-turbo`), análisis IA ≈ 3-5 min, render ≈ 1-3 min por clip. Las re-ejecuciones usan caché en Drive y saltan lo ya calculado.


## 🏗️ Arquitectura

```text
VÍDEO (Drive) ──► COPIA LOCAL ──► ① WHISPER large-v3 · palabra a palabra · VAD Silero
                                  ② AUDIO (librosa): energía, brillo, gritos, picos
                                  ③ SEMÁNTICA: embeddings multilingües + novedad de tema
                                                │
                               ④ FUSIÓN DE SEÑALES por frase (score 0-10)
                                                │
                               ⑤ CANDIDATOS 20/30/45/60 s + anti-solapamiento
                                                │
                               ⑥ DIRECTOR LLM (Gemini): elige, ajusta límites,
                                  títulos, hooks, temas, emojis, score 0-100
                                                │
        ┌──────────────┬──────────────────┬─────┴──────────┬───────────────┐
   ⑦ SILENCIOS     ⑧ REENCUADRE      ⑨ SUBTÍTULOS      ⑩ EFECTOS       ⑪ AUDIO PRO
   EDL + TimeMap   MediaPipe          ASS karaoke       zoom/punch/     denoise+EQ+
   (mapa temporal  seguimiento        con rebote y      shake/flash/    compresor+
   fuente↔salida)  facial suave       emojis PNG        motion blur     loudnorm+ducking
        └──────────────┴──────────────────┴────────────────┴───────────────┘
                                                │
                     ⑫ RENDER (OpenCV frame a frame → tubería ffmpeg)
                            └──► Clips/clip_NNN.mp4 + clip_NNN.json + RESUMEN.txt
```

### Por qué cada tecnología

| Componente | Elección | Motivo |
|---|---|---|
| Transcripción | **faster-whisper (CTranslate2)** | 4-8× más rápido que openai-whisper en T4, timestamps por palabra, VAD Silero integrado (evita alucinaciones en música/silencios) |
| Análisis acústico | **librosa** | RMS/centroide/onsets vectorizados: detecta gritos, energía y excitación en horas de audio en segundos |
| Semántica | **sentence-transformers** (MiniLM multilingüe) | Similitud con "conceptos virales" en español + detección de cambios de tema; ligero y corre en GPU |
| Director editorial | **Gemini** (opcional, gratis) con **respaldo heurístico completo** | El LLM afina límites, títulos y emojis; si no hay key, el pipeline no se degrada, solo pierde finura |
| Reencuadre | **MediaPipe FaceDetection** (respaldo Haar/OpenCV) | Sin descargas de modelos pesados tipo YOLO, preciso para facecams, CPU-friendly; histéresis anti-saltos entre caras |
| Cortes y sincronía | **EDL + TimeMap propio** | Un único mapa temporal fuente↔salida mantiene sincronizados vídeo, audio, subtítulos, emojis y efectos al quitar silencios |
| Subtítulos | **ASS + libass (ffmpeg)** | Tipografía real con karaoke, escalado y colores por palabra: el mismo motor que usan los editores profesionales |
| Efectos y emojis | **OpenCV frame a frame** | Control total (curvas de easing, ducking visual) sin depender de moviepy (lento) |
| Render | **tubería rawvideo → ffmpeg x264** | Un solo encode con subtítulos quemados; audio procesado en paralelo y mux final |

El código vive en el paquete **`clipforge/`** (celdas `%%writefile`): cada módulo es pequeño, testeable y sustituible — arquitectura lista para crecer (diarización pyannote, B-roll, speed-ramps variables…).


In [ ]:
#@title 🧱 1 · Instalación (~2 min)
import os, subprocess, sys, time
t0 = time.time()
os.chdir("/content")
os.makedirs("/content/clipforge", exist_ok=True)
for d in ("/content/clipwork/fonts", "/content/clipwork/emoji", "/content/clipwork/out"):
    os.makedirs(d, exist_ok=True)

print("📦 Instalando paquetes (faster-whisper, sentence-transformers, mediapipe, google-genai)…")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "faster-whisper>=1.0.3", "sentence-transformers", "mediapipe", "google-genai"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("⚠️ pip devolvió errores:\n" + (r.stderr or "")[-800:])

g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
if g.returncode == 0 and g.stdout.strip():
    print("🟢 GPU:", g.stdout.strip())
else:
    print("🔴 SIN GPU → Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU, y ejecuta todo de nuevo.")

import requests
FUENTES = {
    "Poppins ExtraBold": ("Poppins-ExtraBold.ttf", "https://github.com/google/fonts/raw/main/ofl/poppins/Poppins-ExtraBold.ttf"),
    "Anton": ("Anton-Regular.ttf", "https://github.com/google/fonts/raw/main/ofl/anton/Anton-Regular.ttf"),
    "Archivo Black": ("ArchivoBlack-Regular.ttf", "https://github.com/google/fonts/raw/main/ofl/archivoblack/ArchivoBlack-Regular.ttf"),
}
for fam, (fn, url) in FUENTES.items():
    dest = f"/content/clipwork/fonts/{fn}"
    if os.path.exists(dest):
        continue
    try:
        rq = requests.get(url, timeout=30)
        rq.raise_for_status()
        open(dest, "wb").write(rq.content)
        print(f"🔤 Fuente lista: {fam}")
    except Exception as e:
        print(f"⚠️ No se pudo descargar la fuente {fam}: {e}")
print(f"✅ Instalación completa en {time.time() - t0:.0f}s")


## 📁 1b · Código fuente del paquete `clipforge`

Las siguientes celdas escriben los módulos del paquete en `/content/clipforge/`. Si editas una celda de módulo, vuelve a ejecutarla y después reinicia el entorno (`Entorno de ejecución ▸ Reiniciar sesión`) o usa `importlib.reload(...)` para que el cambio se aplique.

| Módulo | Responsabilidad |
|---|---|
| `utils` | subprocesos, ffprobe, cronómetros, colores |
| `transcriber` | Whisper palabra a palabra + frases + caché |
| `audio_features` | energía, brillo, gritos, picos de excitación |
| `semantics` | embeddings, similitud viral, novedad de tema |
| `scoring` | fusión de señales + generación de candidatos |
| `llm_director` | Gemini (títulos/emojis/score) + respaldo heurístico |
| `cuts` | eliminación de silencios (EDL) + TimeMap |
| `reframe` | seguimiento facial y trayectoria de encuadre |
| `subtitles` | ASS karaoke animado estilo Opus Clip |
| `emojis` | PNG Noto Emoji + animación pop |
| `effects` | zoom / punch / shake / flash / motion blur |
| `renderer` | render frame a frame + audio pro + mux |
| `exporter` | clips numerados + JSON de metadatos + resumen |
| `diarization` | [roadmap] hablantes con pyannote |


In [ ]:
#@title 📁 Preparar directorios del paquete
import os, sys
os.chdir("/content")
os.makedirs("/content/clipforge", exist_ok=True)
for d in ("/content/clipwork/fonts", "/content/clipwork/emoji", "/content/clipwork/out"):
    os.makedirs(d, exist_ok=True)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
print("✅ Directorios listos")


In [ ]:
%%writefile clipforge/__init__.py
"""ClipForge AI · motor de clips virales para Google Colab."""
__version__ = "1.0.1"


In [ ]:
%%writefile clipforge/utils.py
"""Utilidades comunes: subprocesos, ffprobe, tiempos y colores."""
import json
import subprocess
import time


def run(cmd, **kw):
    """Ejecuta un comando y lanza un error legible si falla."""
    cmd = [str(c) for c in cmd]
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError("Comando fallido: " + " ".join(cmd)[:300] +
                           "\n--- stderr ---\n" + (r.stderr or "")[-1500:])
    return r


def ffprobe_info(path):
    """Resolución, fps, duración y tamaño de un vídeo."""
    r = run(["ffprobe", "-v", "error", "-print_format", "json",
             "-show_format", "-show_streams", path])
    d = json.loads(r.stdout)
    v = next(s for s in d["streams"] if s["codec_type"] == "video")
    try:
        num, den = (v.get("avg_frame_rate") or "30/1").split("/")
        fps = float(num) / float(den) if float(den) else 30.0
    except Exception:
        fps = 30.0
    if not 5 < fps < 121:
        fps = 30.0
    return {"w": int(v["width"]), "h": int(v["height"]), "fps": fps,
            "dur": float(d["format"]["duration"]), "bytes": int(d["format"]["size"])}


def t2str(t):
    """Segundos → 'H:MM:SS' o 'MM:SS'."""
    t = max(0, int(round(t)))
    h, m, s = t // 3600, t % 3600 // 60, t % 60
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"


class Paso:
    """Context manager que cronometra y muestra el estado de cada paso."""

    def __init__(self, nombre):
        self.nombre = nombre

    def __enter__(self):
        print(f"▶ {self.nombre}…", flush=True)
        self.t0 = time.time()
        return self

    def __exit__(self, exc_type, *a):
        icono = "✅" if exc_type is None else "❌"
        print(f"{icono} {self.nombre} · {time.time() - self.t0:.1f}s", flush=True)


def hex_rgb(hx):
    hx = hx.lstrip("#")
    return int(hx[0:2], 16), int(hx[2:4], 16), int(hx[4:6], 16)


def hex_ass(hx):
    """'#RRGGBB' → color ASS '&H00BBGGRR'."""
    r, g, b = hex_rgb(hx)
    return f"&H00{b:02X}{g:02X}{r:02X}"


In [ ]:
%%writefile clipforge/transcriber.py
"""Transcripción con faster-whisper (CTranslate2) + timestamps por palabra.

Usa el VAD de Silero integrado para saltar zonas sin voz y evitar
alucinaciones en música/silencios. Cachea el resultado en Drive con una
clave (tamaño del vídeo + modelo) para no recalcular en re-ejecuciones.
"""
import json
import re
from pathlib import Path


def _preparar_cuda():
    """Pre-carga cuDNN/cuBLAS instaladas por pip para que CTranslate2 las
    encuentre (fallo clásico de faster-whisper en Colab)."""
    import ctypes
    import glob
    for pat in ("/usr/local/lib/python3*/dist-packages/nvidia/cublas/lib/*.so*",
                "/usr/local/lib/python3*/dist-packages/nvidia/cudnn/lib/*.so*"):
        for so in sorted(glob.glob(pat)):
            try:
                ctypes.CDLL(so)
            except OSError:
                pass


def transcribir(wav, modelo="large-v3", idioma="es", cache=None, forzar=False,
                clave_video=None, progreso=None):
    """Devuelve (palabras, info) con palabras = [{'w','s','e','p'}, ...]."""
    cache = Path(cache) if cache else None
    if cache and cache.exists() and not forzar:
        try:
            d = json.loads(cache.read_text(encoding="utf-8"))
            if d.get("clave") == clave_video and d.get("modelo") == modelo:
                print(f"   💾 Transcripción cargada de caché ({len(d['palabras'])} palabras)")
                return d["palabras"], d["info"]
        except Exception:
            pass
    _preparar_cuda()
    from faster_whisper import WhisperModel
    wm = None
    for dev, ct in (("cuda", "float16"), ("cuda", "int8_float16"), ("cpu", "int8")):
        try:
            print(f"   Cargando Whisper {modelo} en {dev} ({ct})…")
            wm = WhisperModel(modelo, device=dev, compute_type=ct)
            break
        except Exception as e:
            print(f"   ⚠️ {dev}/{ct}: {str(e)[:120]}")
    if wm is None:
        raise RuntimeError("No se pudo cargar Whisper en ningún dispositivo")
    segs, inf = wm.transcribe(wav, language=idioma or None, vad_filter=True,
                              word_timestamps=True, beam_size=5)
    palabras = []
    for seg in segs:
        for w in (seg.words or []):
            t = w.word.strip()
            if t:
                palabras.append({"w": t, "s": round(w.start, 3),
                                 "e": round(w.end, 3), "p": round(w.probability, 3)})
        if progreso:
            progreso(min(seg.end, inf.duration), inf.duration)
    info = {"dur": inf.duration, "idioma": inf.language, "n": len(palabras)}
    if cache:
        cache.write_text(json.dumps({"clave": clave_video, "modelo": modelo,
                                     "info": info, "palabras": palabras},
                                    ensure_ascii=False), encoding="utf-8")
    return palabras, info


_FIN = re.compile(r"[\.\?\!…]$")


def construir_frases(palabras, gap_corte=0.8, max_palabras=42):
    """Agrupa palabras en frases usando puntuación y pausas."""
    frases, cur = [], []
    for i, w in enumerate(palabras):
        cur.append(w)
        gap = palabras[i + 1]["s"] - w["e"] if i + 1 < len(palabras) else 99.0
        if _FIN.search(w["w"]) or gap > gap_corte or len(cur) >= max_palabras:
            frases.append(_cerrar(cur, len(frases)))
            cur = []
    if cur:
        frases.append(_cerrar(cur, len(frases)))
    return frases


def _cerrar(ws, i):
    return {"i": i, "texto": " ".join(x["w"] for x in ws),
            "s": ws[0]["s"], "e": ws[-1]["e"], "n": len(ws)}


In [ ]:
%%writefile clipforge/audio_features.py
"""Análisis acústico del directo con librosa.

Extrae, en una rejilla temporal de 0.25 s:
- rms_db  : volumen (subidas de energía)
- arousal : "excitación" 0-1 (mezcla de energía + brillo espectral + onsets)
- gritos  : exceso de energía sobre el nivel habitual (picos fuertes)
"""
import numpy as np
from pathlib import Path

HOP_S = 0.25


def calcular(wav, cache=None, forzar=False):
    cache = Path(cache) if cache else None
    if cache and cache.exists() and not forzar:
        try:
            d = np.load(cache)
            feat = {k: d[k] for k in d.files}
            print(f"   💾 Rasgos de audio cargados de caché ({len(feat['t'])} ventanas)")
            return feat
        except Exception:
            pass
    import librosa
    y, sr = librosa.load(wav, sr=16000, mono=True)
    hop = int(sr * HOP_S)
    rms = librosa.feature.rms(y=y, frame_length=hop * 2, hop_length=hop)[0]
    rms_db = librosa.amplitude_to_db(rms + 1e-9)
    cen = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
    ons = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop)
    n = min(len(rms_db), len(cen), len(ons))
    rms_db, cen, ons = rms_db[:n], cen[:n], ons[:n]
    t = np.arange(n) * HOP_S

    def z_robusta(x):
        med = np.median(x)
        mad = np.median(np.abs(x - med)) + 1e-9
        return (x - med) / (1.4826 * mad)

    def suave(x, k=5):
        return np.convolve(x, np.ones(k) / k, mode="same")

    arousal = 0.55 * z_robusta(rms_db) + 0.25 * z_robusta(cen) + 0.20 * z_robusta(ons)
    arousal = np.clip(suave(arousal) * 0.32 + 0.5, 0, 1)
    gritos = np.clip(z_robusta(rms_db) - 1.8, 0, None)
    feat = {"t": t, "rms_db": rms_db, "arousal": arousal, "gritos": gritos}
    if cache:
        np.savez_compressed(cache, **feat)
    return feat


def stat(feat, campo, t0, t1, fn="mean"):
    """Estadístico de una serie en el rango temporal [t0, t1]."""
    i0, i1 = np.searchsorted(feat["t"], [t0, t1])
    seg = feat[campo][max(0, i0):max(i0 + 1, i1)]
    if len(seg) == 0:
        return 0.0
    return float(seg.max() if fn == "max" else seg.mean())


def picos(feat, min_sep=8.0, umbral=0.62):
    """Tiempos de picos de excitación separados al menos min_sep segundos."""
    a, t = feat["arousal"], feat["t"]
    out = []
    for i in np.argsort(a)[::-1]:
        if a[i] < umbral:
            break
        if all(abs(t[i] - x) >= min_sep for x in out):
            out.append(float(t[i]))
    return sorted(out)


In [ ]:
%%writefile clipforge/semantics.py
"""Análisis semántico con sentence-transformers (multilingüe).

- viralidad: similitud de cada frase con "conceptos virales" (polémica,
  humor, dato sorprendente, exclusiva…)
- novedad  : distancia respecto al contexto anterior (cambios de tema)
"""
import numpy as np

MODELO = "paraphrase-multilingual-MiniLM-L12-v2"

CONSULTAS_VIRALES = [
    "una polémica o un escándalo muy fuerte",
    "una opinión contundente y sin filtros",
    "un dato sorprendente que nadie esperaba",
    "un momento muy gracioso y divertido",
    "una noticia importante de última hora",
    "una discusión o un enfrentamiento tenso",
    "una frase épica, lapidaria y memorable",
    "una crítica dura a alguien famoso",
    "dinero, cifras enormes y récords",
    "un anuncio o revelación exclusiva",
]

_modelo = None


def _cargar():
    global _modelo
    if _modelo is None:
        from sentence_transformers import SentenceTransformer
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        _modelo = SentenceTransformer(MODELO, device=dev)
    return _modelo


def calcular(textos):
    """Embeddings L2-normalizados de todas las frases."""
    m = _cargar()
    return m.encode(textos, batch_size=128, convert_to_numpy=True,
                    normalize_embeddings=True, show_progress_bar=True)


def viralidad(emb):
    """Máxima similitud de cada frase con los conceptos virales, 0-1."""
    q = _cargar().encode(CONSULTAS_VIRALES, convert_to_numpy=True,
                         normalize_embeddings=True)
    v = (emb @ q.T).max(axis=1)
    return (v - v.min()) / (v.max() - v.min() + 1e-9)


def novedad(emb, k=6):
    """1 - similitud con la media de las k frases previas (cambio de tema)."""
    n = len(emb)
    out = np.zeros(n)
    for i in range(1, n):
        ctx = emb[max(0, i - k):i].mean(axis=0)
        ctx /= (np.linalg.norm(ctx) + 1e-9)
        out[i] = 1.0 - float(emb[i] @ ctx)
    if out.max() > 0:
        out = np.clip(out / (np.percentile(out, 95) + 1e-9), 0, 1)
    return out


In [ ]:
%%writefile clipforge/scoring.py
"""Puntuación multi-señal de frases y generación de clips candidatos.

Cada frase recibe un score 0-10 que combina:
  kw     · palabras clave virales en español      punct · ¡! y ¿?
  audio  · excitación acústica (energía, gritos)  ritmo · velocidad al hablar
  sem    · similitud semántica con conceptos virales (embeddings)
  nov    · novedad / cambio de tema

Los candidatos son ventanas de frases ajustadas a duraciones objetivo
(20/30/45/60 s) y se filtran por solapamiento.
"""
import math
import re
import numpy as np
from . import audio_features as af

PESOS = {"kw": 1.0, "audio": 1.3, "sem": 1.1, "nov": 0.6, "punct": 0.5, "ritmo": 0.4}

PALABRAS_CLAVE = {
    # impacto / sorpresa
    "increíble": 3, "increible": 3, "brutal": 3, "locura": 3, "tremendo": 3,
    "bestial": 3, "alucinante": 3, "flipante": 3, "flipa": 3, "flipando": 3,
    "espectacular": 2, "impresionante": 2, "no puede ser": 4, "no me lo creo": 3,
    "madre mía": 3, "ojo": 2, "atención": 2, "escucha": 2, "bomba": 4, "bombazo": 4,
    # polémica / conflicto
    "polémica": 4, "polémico": 3, "escándalo": 4, "vergüenza": 3, "vergonzoso": 3,
    "desastre": 3, "ridículo": 3, "culpa": 2, "mentira": 3, "engaño": 3,
    "guerra": 2, "ataque": 2, "golpe": 3, "hostia": 3, "joder": 2, "cabreo": 3,
    # datos / dinero
    "récord": 3, "histórico": 3, "millones": 3, "dinero": 3, "euros": 2,
    "secreto": 3, "exclusiva": 3, "confirmado": 3, "oficial": 3, "última hora": 4,
    # narrativa
    "nunca": 2, "jamás": 2, "nadie": 2, "mejor": 2, "peor": 2,
    "verdad": 2, "importante": 2, "clave": 2, "gravísimo": 3, "grave": 2,
    # motor / MotoGP (contexto del canal)
    "adelantamiento": 2, "accidente": 3, "caída": 3, "sanción": 3, "multa": 2,
    "descalificado": 3, "pole": 2, "victoria": 2, "campeón": 2, "mundial": 2, "podio": 2,
    "márquez": 2, "marquez": 2, "bagnaia": 2, "pecco": 2, "acosta": 2,
    "quartararo": 2, "martín": 2, "viñales": 2, "rossi": 2, "lorenzo": 2,
    "ducati": 2, "yamaha": 2, "honda": 2, "ktm": 2, "aprilia": 2, "motogp": 2,
}


def _kw(texto):
    t = " " + texto.lower() + " "
    return sum(p for k, p in PALABRAS_CLAVE.items() if k in t)


def puntuar_frases(frases, feat, viral, nov):
    """Añade a cada frase su 'score' 0-10 y el desglose por señal."""
    for f in frases:
        dur = max(f["e"] - f["s"], 0.4)
        c = {
            "kw": min(_kw(f["texto"]) / 6.0, 1.0),
            "audio": min(af.stat(feat, "arousal", f["s"], f["e"]) +
                         0.5 * af.stat(feat, "gritos", f["s"], f["e"], "max"), 1.2),
            "sem": float(viral[f["i"]]),
            "nov": float(nov[f["i"]]),
            "punct": min((f["texto"].count("!") + f["texto"].count("?")) / 2.0, 1.0),
            "ritmo": 1 / (1 + math.exp(-(f["n"] / dur - 3.2))),
        }
        f["score"] = round(sum(PESOS[k] * v for k, v in c.items()) / sum(PESOS.values()) * 10, 2)
        f["senales"] = {k: round(v, 3) for k, v in c.items()}
    return frases


_TERMINAL = re.compile(r"[\.\?\!…]$")


def candidatos(frases, num, duraciones=(20, 30, 45, 60), dur_min=18, dur_max=68, umbral=2.0):
    """Genera ventanas candidatas y devuelve las mejores sin solaparse.

    Para cada frase de arranque prueba las 4 duraciones objetivo y se queda
    con la mejor: así el sistema decide la duración según el contenido.
    """
    cands = []
    N = len(frases)
    for i in range(N):
        if frases[i]["score"] < umbral:   # el gancho inicial debe ser potente
            continue
        mejor = None
        for objetivo in duraciones:
            k = i
            while k < N and frases[k]["e"] - frases[i]["s"] <= dur_max:
                dur = frases[k]["e"] - frases[i]["s"]
                if dur >= dur_min and abs(dur - objetivo) <= objetivo * 0.25:
                    span = frases[i:k + 1]
                    media = sum(f["score"] for f in span) / len(span)
                    maxi = max(f["score"] for f in span)
                    gancho = sum(f["score"] for f in span[:2]) / min(2, len(span))
                    sc = 0.5 * media + 0.25 * maxi + 0.25 * gancho
                    if _TERMINAL.search(frases[k]["texto"]):
                        sc += 0.35                      # cierre natural
                    sc -= abs(dur - objetivo) / objetivo * 0.4
                    if mejor is None or sc > mejor[0]:
                        mejor = (sc, i, k, dur, objetivo)
                k += 1
        if mejor:
            cands.append(mejor)
    cands.sort(key=lambda x: -x[0])
    eleg = []
    for sc, i, k, dur, obj in cands:
        s, e = frases[i]["s"], frases[k]["e"]
        if any(min(e, x["end"]) - max(s, x["start"]) > 0.2 * dur for x in eleg):
            continue
        eleg.append({"start": round(s, 2), "end": round(e, 2), "dur": round(dur, 1),
                     "objetivo": obj, "i0": i, "i1": k,
                     "score": int(np.clip(sc * 10 + 12, 5, 99))})
        if len(eleg) >= num:
            break
    eleg.sort(key=lambda c: c["start"])
    return eleg


In [ ]:
%%writefile clipforge/llm_director.py
"""Director editorial con LLM (Gemini) + respaldo heurístico completo.

El LLM recibe los candidatos con su transcripción, elige los mejores,
puede ajustar los límites (±4 frases) y genera título, hook, tema,
resumen, keywords, hashtags, emojis con cita exacta y score 0-100.
Si no hay API key o el servicio falla, todo se genera heurísticamente:
el pipeline nunca se detiene.
"""
import json
import os
import re
from collections import Counter

MODELOS = ("gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-flash")

STOP = set(("de la que el en y a los se del las un por con no una su para es al lo como "
            "más pero sus le ya o fue este ha sí porque esta son entre cuando muy sin sobre "
            "ser tiene también me hasta hay donde quien desde todo nos durante todos uno les "
            "ni contra otros ese eso ante ellos e esto mí antes algunos qué unos yo otro otras "
            "otra él tanto esa estos mucho quienes nada muchos cual poco ella estar estas "
            "algunas algo nosotros vamos estamos tiene tenemos porque entonces bueno pues").split())

PLANTILLA = """Eres el director editorial de clips virales de un canal de YouTube.
Contexto del canal: <<CTX>>

Te paso <<NC>> momentos candidatos de un directo, con sus frases transcritas.
Elige los <<N>> MEJORES para TikTok/Shorts/Reels y devuelve SOLO JSON válido.

Para cada clip elegido:
- "id": id del candidato.
- "usar": true.
- "ajuste_ini" / "ajuste_fin": desplazamiento en nº de frases (-4 a 4) si mejora el gancho o el cierre.
- "titulo": español, viral, máx 70 caracteres, 1-2 emojis, sin comillas.
- "hook": rótulo corto para sobreimprimir en el vídeo, máx 45 caracteres, SIN emojis.
- "tema": 2-4 palabras.
- "resumen": máx 180 caracteres.
- "keywords": 5 palabras clave.
- "hashtags": 5 hashtags en español.
- "score": viralidad 0-100 (sé exigente, no regales puntos).
- "emojis": 2-5 objetos {"quote": "2-4 palabras EXACTAS de la transcripción", "emoji": "🔥"}.

Criterios: gancho en los 2 primeros segundos, polémica, opiniones fuertes,
datos sorprendentes, humor, tensión, cierre natural. Descarta lo aburrido.

Formato de respuesta:
{"clips": [{"id": 3, "usar": true, "ajuste_ini": 0, "ajuste_fin": 1, "titulo": "…", "hook": "…", "tema": "…", "resumen": "…", "keywords": ["…"], "hashtags": ["#…"], "score": 87, "emojis": [{"quote": "…", "emoji": "🔥"}]}]}

CANDIDATOS:
<<CANDS>>"""


def clave_api():
    try:
        from google.colab import userdata
        k = userdata.get("GEMINI_API_KEY")
        if k:
            return k
    except Exception:
        pass
    return os.environ.get("GEMINI_API_KEY", "")


def _texto_candidatos(cands, frases):
    bloques = []
    for n, c in enumerate(cands):
        frs = frases[c["i0"]:c["i1"] + 1]
        txt = " ".join(f["texto"] for f in frs)[:900]
        m0, s0 = divmod(int(c["start"]), 60)
        bloques.append(f"[id {n} · min {m0:02d}:{s0:02d} · {c['dur']:.0f}s · score base {c['score']}]\n{txt}")
    return "\n\n".join(bloques)


def _extraer_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.M).strip()
    m = re.search(r"\{[\s\S]*\}", texto)
    return json.loads(m.group(0) if m else texto)


def dirigir(cands, frases, num, contexto="", canal="PecinoGP"):
    """Devuelve el plan final de clips con metadatos completos."""
    plan = None
    key = clave_api()
    if key:
        plan = _con_gemini(cands, frases, num, contexto, key)
    if plan is None:
        print("   ℹ️ Selección y títulos heurísticos. Añade GEMINI_API_KEY en los "
              "Secretos 🔑 de Colab (gratis: aistudio.google.com/app/apikey) para mejorarlos.")
        plan = []
    for c in sorted(cands, key=lambda x: -x["score"]):
        if len(plan) >= num:
            break
        if any(min(c["end"], p["end"]) - max(c["start"], p["start"]) > 3 for p in plan):
            continue
        plan.append(_meta_heuristica(c, frases, canal))
    plan = sorted(plan[:num], key=lambda c: c["start"])
    for j, c in enumerate(plan, 1):
        c["id"] = j
    return plan


def _con_gemini(cands, frases, num, contexto, key):
    prompt = (PLANTILLA.replace("<<CTX>>", contexto).replace("<<N>>", str(num))
              .replace("<<NC>>", str(len(cands)))
              .replace("<<CANDS>>", _texto_candidatos(cands, frases)))
    try:
        from google import genai
        cli = genai.Client(api_key=key)
    except Exception as e:
        print(f"   ⚠️ google-genai no disponible: {e}")
        return None
    for modelo in MODELOS:
        try:
            print(f"   🤖 Analizando candidatos con {modelo}…")
            r = cli.models.generate_content(
                model=modelo, contents=prompt,
                config={"response_mime_type": "application/json", "temperature": 0.4})
            data = _extraer_json(r.text or "")
            plan = []
            for c in data.get("clips", []):
                if not c.get("usar", True):
                    continue
                base = cands[int(c["id"])]
                i0 = max(0, min(len(frases) - 1, base["i0"] + int(c.get("ajuste_ini", 0) or 0)))
                i1 = max(i0, min(len(frases) - 1, base["i1"] + int(c.get("ajuste_fin", 0) or 0)))
                s, e = frases[i0]["s"], frases[i1]["e"]
                if not 12 <= e - s <= 90:
                    i0, i1, s, e = base["i0"], base["i1"], base["start"], base["end"]
                plan.append({
                    "start": round(s, 2), "end": round(e, 2), "dur": round(e - s, 1),
                    "i0": i0, "i1": i1,
                    "score": int(round(0.55 * int(c.get("score", base["score"])) + 0.45 * base["score"])),
                    "score_llm": int(c.get("score", 0)), "score_heur": base["score"],
                    "titulo": str(c.get("titulo", "")).strip()[:80] or "🔥 Momentazo del directo",
                    "hook": str(c.get("hook", "")).strip()[:48],
                    "tema": str(c.get("tema", "directo"))[:40],
                    "resumen": str(c.get("resumen", ""))[:220],
                    "keywords": [str(x) for x in c.get("keywords", [])][:6],
                    "hashtags": [str(x) for x in c.get("hashtags", [])][:6],
                    "emojis": [x for x in c.get("emojis", []) if isinstance(x, dict)][:5],
                    "fuente": modelo,
                })
            if plan:
                plan.sort(key=lambda p: p["start"])
                limpio = []
                for p in plan:
                    if limpio and p["start"] < limpio[-1]["end"] - 2:
                        if p["score"] > limpio[-1]["score"]:
                            limpio[-1] = p
                        continue
                    limpio.append(p)
                return limpio[:num]
        except Exception as e:
            print(f"   ⚠️ {modelo}: {type(e).__name__} {str(e)[:150]}")
    return None


def _meta_heuristica(c, frases, canal):
    span = frases[c["i0"]:c["i1"] + 1]
    top = max(span, key=lambda f: f["score"])
    titulo = top["texto"].strip().rstrip(".")
    titulo = (titulo[:62] + "…") if len(titulo) > 64 else titulo
    tokens = [t for t in re.findall(r"[a-záéíóúñü]{4,}",
                                    " ".join(f["texto"] for f in span).lower())
              if t not in STOP]
    kws = [w for w, _ in Counter(tokens).most_common(5)]
    out = dict(c)
    out.update({
        "titulo": "🔥 " + titulo.capitalize(),
        "hook": titulo[:45].upper(),
        "tema": kws[0] if kws else "directo",
        "resumen": " ".join(f["texto"] for f in span)[:200],
        "keywords": kws,
        "hashtags": ["#shorts", "#viral", "#motogp", f"#{canal.lower()}", "#directo"],
        "emojis": [],
        "fuente": "heuristica",
    })
    return out


In [ ]:
%%writefile clipforge/cuts.py
"""Eliminación de silencios y mapa temporal fuente→salida.

Construye una EDL (lista de rangos que se conservan) a partir de los
timestamps de palabra: las pausas más largas que `max_pausa` se comprimen.
TimeMap traduce tiempos en ambos sentidos (incluida la velocidad global)
para mantener sincronizados vídeo, audio, subtítulos, emojis y efectos.
"""
import numpy as np


def construir_edl(palabras, ini, fin, quitar=True, max_pausa=0.7,
                  pausa_restante=0.24, margen=0.12):
    ws = [w for w in palabras if w["s"] < fin and w["e"] > ini]
    if not quitar or not ws:
        return [(ini, fin)]
    rangos = []
    cur_s = max(ini, ws[0]["s"] - margen * 2)
    cur_e = ws[0]["e"]
    for w in ws[1:]:
        if w["s"] - cur_e > max_pausa:
            rangos.append((cur_s, min(fin, cur_e + pausa_restante / 2)))
            cur_s = max(ini, w["s"] - pausa_restante / 2)
        cur_e = w["e"]
    rangos.append((cur_s, min(fin, cur_e + margen * 3)))
    out = []
    for s, e in rangos:                      # fusionar contiguos, quitar migajas
        s, e = max(ini, s), min(fin, e)
        if out and s - out[-1][1] < 0.12:
            out[-1] = (out[-1][0], e)
        elif e - s > 0.15:
            out.append((s, e))
    return out or [(ini, fin)]


class TimeMap:
    """Mapa fuente↔salida sobre una EDL, con velocidad global opcional."""

    def __init__(self, edl, velocidad=1.0):
        self.edl = edl
        self.v = max(0.5, min(2.0, velocidad))
        src, out, acc = [], [], 0.0
        for s, e in edl:
            src += [s, e]
            out += [acc, acc + (e - s) / self.v]
            acc = out[-1]
        self.src_k = np.array(src)
        self.out_k = np.array(out)
        self.out_dur = acc

    def out2src(self, t):
        return float(np.interp(t, self.out_k, self.src_k))

    def src2out(self, t):
        # un tiempo dentro de un hueco eliminado cae en el corte
        return float(np.interp(t, self.src_k, self.out_k))

    def remap_palabras(self, palabras):
        out = []
        for w in palabras:
            for s, e in self.edl:
                if w["e"] > s and w["s"] < e:
                    a = self.src2out(max(w["s"], s))
                    b = self.src2out(min(w["e"], e))
                    if b - a > 0.03:
                        out.append({"w": w["w"], "s": round(a, 3), "e": round(b, 3)})
                    break
        return out


In [ ]:
%%writefile clipforge/reframe.py
"""Reencuadre inteligente 16:9 → 9:16 con seguimiento facial.

MediaPipe FaceDetection (con respaldo Haar de OpenCV) muestrea caras a lo
largo del clip. La cara principal se elige con histéresis (no saltar entre
caras: solo cambia si otra es claramente mayor durante varias muestras),
se rellenan huecos manteniendo el último objetivo y la trayectoria se
suaviza con un filtro gaussiano + límite de velocidad de paneo.
"""
import cv2
import numpy as np

_detector = None


def _cargar_detector():
    global _detector
    if _detector is None:
        try:
            import mediapipe as mp
            fd = mp.solutions.face_detection.FaceDetection(
                model_selection=1, min_detection_confidence=0.4)
            _detector = ("mediapipe", fd)
        except Exception:
            haar = cv2.CascadeClassifier(
                cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
            _detector = ("haar", haar)
            print("   ⚠️ MediaPipe no disponible → usando detector Haar de OpenCV")
    return _detector


def detectar_caras(frame_bgr):
    """Lista de caras [(cx, cy, w, h)] en coordenadas relativas 0-1."""
    tipo, det = _cargar_detector()
    H, W = frame_bgr.shape[:2]
    if tipo == "mediapipe":
        res = det.process(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        out = []
        for d in (res.detections or []):
            bb = d.location_data.relative_bounding_box
            out.append((bb.xmin + bb.width / 2, bb.ymin + bb.height / 2, bb.width, bb.height))
        return out
    g = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    caras = det.detectMultiScale(g, 1.1, 5, minSize=(max(24, W // 25), max(24, W // 25)))
    return [((x + w / 2) / W, (y + h / 2) / H, w / W, h / H) for x, y, w, h in caras]


def calcular_ruta(src, tm, W, H, fps_muestreo=2.5, modo="auto"):
    """Trayectoria de encuadre en tiempo de SALIDA: dict(t, cx, cy, ratio)."""
    dur = tm.out_dur
    n = max(4, int(dur * fps_muestreo))
    t_out = np.linspace(0, max(dur - 0.05, 0.1), n)
    cx = np.full(n, 0.5)
    cy = np.full(n, 0.46)
    hallazgos = 0
    if modo != "centro":
        cap = cv2.VideoCapture(src)
        objetivo = None
        paciencia = 0
        for i, to in enumerate(t_out):
            cap.set(cv2.CAP_PROP_POS_MSEC, tm.out2src(float(to)) * 1000.0)
            ok, fr = cap.read()
            if not ok:
                continue
            if fr.shape[1] > 640:
                fr = cv2.resize(fr, (640, int(fr.shape[0] * 640 / fr.shape[1])))
            caras = detectar_caras(fr)
            if caras:
                hallazgos += 1
                grande = max(caras, key=lambda c: c[2] * c[3])
                if objetivo is None:
                    objetivo = grande
                else:
                    cerca = [c for c in caras if abs(c[0] - objetivo[0]) < 0.13]
                    if cerca:
                        cand = max(cerca, key=lambda c: c[2] * c[3])
                        if grande[2] * grande[3] > cand[2] * cand[3] * 1.8:
                            paciencia += 1          # otra cara domina: esperar
                            if paciencia >= 3:
                                cand, paciencia = grande, 0
                        else:
                            paciencia = 0
                        objetivo = cand
                    else:
                        objetivo = grande
            if objetivo is not None:
                cx[i], cy[i] = objetivo[0], objetivo[1]
        cap.release()
    if n > 5:                                   # suavizado + límite de paneo
        sig = max(1, int(0.8 * fps_muestreo))
        ker = np.exp(-0.5 * (np.arange(-3 * sig, 3 * sig + 1) / sig) ** 2)
        ker /= ker.sum()
        for arr in (cx, cy):
            pad = np.pad(arr, 3 * sig, mode="edge")
            arr[:] = np.convolve(pad, ker, mode="same")[3 * sig:-3 * sig]
        vmax = 0.10 / fps_muestreo              # máx. 10% del ancho por segundo
        for i in range(1, n):
            cx[i] = cx[i - 1] + np.clip(cx[i] - cx[i - 1], -vmax, vmax)
            cy[i] = cy[i - 1] + np.clip(cy[i] - cy[i - 1], -vmax, vmax)
    return {"t": t_out, "cx": cx * W, "cy": cy * H, "ratio": hallazgos / n}


In [ ]:
%%writefile clipforge/subtitles.py
"""Subtítulos ASS estilo Opus Clip / CapCut / Submagic:

- páginas de máx. 3 palabras (nunca llenan la pantalla)
- karaoke palabra a palabra: la palabra activa se resalta en color con
  rebote (escala 86 → 118 → 100) y el resto queda en blanco
- blanco + borde negro grueso + sombra, tamaño adaptativo
- entrada/salida suaves por página, rótulo superior (hook) y marca de agua
"""
import re

_EMOJI = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF\u2300-\u23FF\u2B00-\u2BFF\u2190-\u21FF\u2122\u00A9\u00AE\uFE0F\u200D]")


def _limpiar(t):
    t = _EMOJI.sub("", t).replace("{", "(").replace("}", ")").replace("\\", "/")
    return re.sub(r"\s+", " ", t).strip()


def _t(t):
    t = max(0.0, t)
    h = int(t // 3600)
    m = int(t % 3600 // 60)
    return f"{h}:{m:02d}:{t % 60:05.2f}"


def _hex_ass(hx):
    hx = hx.lstrip("#")
    return "&H00" + hx[4:6].upper() + hx[2:4].upper() + hx[0:2].upper()


def paginar(palabras, max_pal=3, max_dur=1.6, corte_gap=0.6):
    pags, cur = [], []
    for i, w in enumerate(palabras):
        cur.append(w)
        gap = palabras[i + 1]["s"] - w["e"] if i + 1 < len(palabras) else 9.0
        puntua = re.search(r"[\.\?\!…,;:]$", w["w"]) is not None
        if (len(cur) >= max_pal or cur[-1]["e"] - cur[0]["s"] > max_dur
                or gap > corte_gap or (puntua and len(cur) >= 2)):
            pags.append(cur)
            cur = []
    if cur:
        pags.append(cur)
    return pags


def generar_ass(palabras, ruta, dur, fuente="Poppins ExtraBold", color="#FFE600",
                mayusculas=True, hook="", marca="", con_subs=True):
    HL = _hex_ass(color)
    L = ["[Script Info]", "ScriptType: v4.00+", "PlayResX: 1080", "PlayResY: 1920",
         "WrapStyle: 0", "ScaledBorderAndShadow: yes", "", "[V4+ Styles]",
         ("Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, "
          "OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, "
          "ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, "
          "MarginL, MarginR, MarginV, Encoding"),
         f"Style: Cap,{fuente},92,&H00FFFFFF,&H00FFFFFF,&H00101010,&H9B000000,0,0,0,0,100,100,0,0,1,8,3,2,60,60,600,1",
         f"Style: Head,{fuente},54,&H00FFFFFF,&H00FFFFFF,&H87000000,&H87000000,0,0,0,0,100,100,0,0,3,7,0,8,80,80,116,1",
         f"Style: WM,{fuente},34,&H7DFFFFFF,&H00FFFFFF,&H7D101010,&H00000000,0,0,0,0,100,100,0,0,1,2,0,3,40,36,52,1",
         "", "[Events]",
         "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text"]
    if hook:
        L.append(f"Dialogue: 2,{_t(0)},{_t(dur)},Head,,0,0,0,," +
                 "{\\fad(220,220)}" + _limpiar(hook).upper())
    if marca:
        L.append(f"Dialogue: 2,{_t(0)},{_t(dur)},WM,,0,0,0,," + _limpiar(marca))
    if con_subs:
        for pag in paginar(palabras):
            fin_pag = pag[-1]["e"]
            nch = sum(len(w["w"]) for w in pag) + len(pag) - 1
            fs = "{\\fs64}" if nch > 24 else ("{\\fs76}" if nch > 17 else "")
            for j, w in enumerate(pag):
                a = w["s"]
                b = pag[j + 1]["s"] if j + 1 < len(pag) else fin_pag
                if b - a < 0.06:
                    b = a + 0.06
                partes = []
                for k, x in enumerate(pag):
                    tx = _limpiar(re.sub(r"[\.,;:]+$", "", x["w"]))
                    if mayusculas:
                        tx = tx.upper()
                    if not tx:
                        continue
                    if k == j:   # palabra activa: color + rebote
                        partes.append("{\\c" + HL + "\\fscx86\\fscy86"
                                      "\\t(0,70,\\fscx118\\fscy118)"
                                      "\\t(70,150,\\fscx100\\fscy100)}" + tx + "{\\r}")
                    else:
                        partes.append(tx)
                extra = "{\\fad(90,0)}" if j == 0 else ("{\\fad(0,70)}" if j == len(pag) - 1 else "")
                L.append(f"Dialogue: 0,{_t(a)},{_t(b)},Cap,,0,0,0,,{extra}{fs}" + " ".join(partes))
    ruta.write_text("\n".join(L) + "\n", encoding="utf-8")
    return ruta


In [ ]:
%%writefile clipforge/emojis.py
"""Emojis contextuales renderizados como PNG (Noto Emoji) con animación pop.

Se colocan por: (1) citas exactas sugeridas por el LLM, o (2) mapa de
palabras clave. Espaciados ≥2.5 s y limitados por clip para no saturar.
"""
import re
import unicodedata
from pathlib import Path
import cv2
import numpy as np

MAPA = [
    (r"dinero|millon|euro|pasta|caro|precio", "💰"),
    (r"\bjaja|\brisa|gracioso", "😂"),
    (r"incre[ií]ble|brutal|locura|tremend|bestial|flip", "🔥"),
    (r"\bojo\b|cuidado|atenci[oó]n", "👀"),
    (r"golpe|bomba|explot|estall", "💥"),
    (r"error|fallo|desastre|fatal", "❌"),
    (r"\bgan(a|ó|ar)|campe[oó]n|victoria|triunf", "🏆"),
    (r"carrera|circuito|adelanta|\bmoto", "🏍️"),
    (r"r[aá]pid|velocidad|volando", "⚡"),
    (r"verg[uü]enza|enfad|cabre|indignante", "😡"),
    (r"no puede ser|alucin|sorprend|impact", "😱"),
    (r"peligro|riesgo|amenaza", "⚠️"),
    (r"sanci[oó]n|multa|castigo|il[eí]gal", "🚨"),
    (r"objetivo|clave|exacto|justo", "🎯"),
    (r"sube|crece|r[eé]cord|m[aá]ximo", "📈"),
    (r"explota la cabeza|flipas|mental", "🤯"),
]


def _norm(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "", s)


def _codigos(e, sep, con_fe0f=False):
    return sep.join(f"{ord(c):x}" for c in e if con_fe0f or ord(c) != 0xFE0F)


def descargar(e, carpeta):
    """PNG BGRA 220 px del emoji (Noto → Twemoji de respaldo), con caché."""
    carpeta = Path(carpeta)
    destino = carpeta / f"e_{_codigos(e, '_')}.png"
    if destino.exists():
        return str(destino)
    import requests
    urls = [
        f"https://raw.githubusercontent.com/googlefonts/noto-emoji/main/png/512/emoji_u{_codigos(e, '_')}.png",
        f"https://cdn.jsdelivr.net/gh/jdecked/twemoji@15.1.0/assets/72x72/{_codigos(e, '-')}.png",
        f"https://cdn.jsdelivr.net/gh/jdecked/twemoji@15.1.0/assets/72x72/{_codigos(e, '-', True)}.png",
    ]
    for u in urls:
        try:
            r = requests.get(u, timeout=15)
            if r.status_code != 200:
                continue
            img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_UNCHANGED)
            if img is None or img.ndim != 3:
                continue
            if img.shape[2] == 3:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
            img = cv2.resize(img, (220, 220), interpolation=cv2.INTER_LANCZOS4)
            cv2.imwrite(str(destino), img)
            return str(destino)
        except Exception:
            continue
    return None


def momentos(clip, palabras_out, carpeta, activo=True, maximo=6):
    """[(t_salida, ruta_png)] para un clip."""
    if not activo or not palabras_out:
        return []
    brutos = []
    for m in clip.get("emojis", []) or []:      # citas del LLM
        cita, emo = str(m.get("quote", "")), str(m.get("emoji", "")).strip()
        if not cita or not emo:
            continue
        primera = _norm(cita.split()[0]) if cita.split() else ""
        for w in palabras_out:
            if primera and _norm(w["w"]) == primera:
                brutos.append((w["s"], emo))
                break
    if not brutos:                              # respaldo: mapa de keywords
        for w in palabras_out:
            bajo = w["w"].lower()
            for rx, emo in MAPA:
                if re.search(rx, bajo):
                    brutos.append((w["s"], emo))
                    break
    brutos.sort()
    out, previo = [], -9.0
    for t, e in brutos:
        if t < 1.0 or t - previo < 2.5:
            continue
        png = descargar(e, carpeta)
        if png:
            out.append((float(t), png))
            previo = t
        if len(out) >= maximo:
            break
    return out


def _ease_out_back(p):
    c1, c3 = 1.70158, 2.70158
    return 1 + c3 * (p - 1) ** 3 + c1 * (p - 1) ** 2


def pegar(frame, png_bgra, cx, cy, escala=1.0, alfa=1.0):
    """Pega un PNG BGRA sobre el frame con mezcla alfa."""
    if escala <= 0.02 or alfa <= 0.02:
        return
    h, w = png_bgra.shape[:2]
    nw, nh = max(2, int(w * escala)), max(2, int(h * escala))
    img = cv2.resize(png_bgra, (nw, nh), interpolation=cv2.INTER_LINEAR)
    x0, y0 = int(cx - nw / 2), int(cy - nh / 2)
    fx0, fy0 = max(0, x0), max(0, y0)
    fx1, fy1 = min(frame.shape[1], x0 + nw), min(frame.shape[0], y0 + nh)
    if fx1 <= fx0 or fy1 <= fy0:
        return
    sub = img[fy0 - y0:fy1 - y0, fx0 - x0:fx1 - x0].astype(np.float32)
    a = sub[:, :, 3:4] / 255.0 * alfa
    roi = frame[fy0:fy1, fx0:fx1].astype(np.float32)
    frame[fy0:fy1, fx0:fx1] = (roi * (1 - a) + sub[:, :, :3] * a).astype(np.uint8)


def dibujar(frame, png, t_rel):
    """Anima el emoji: pop con rebote (0.28 s) → visible → fundido de salida."""
    DUR = 2.0
    if not 0 <= t_rel < DUR:
        return
    if t_rel < 0.28:
        esc = max(0.05, _ease_out_back(t_rel / 0.28))
        alfa = min(1.0, t_rel / 0.10)
    elif t_rel > DUR - 0.25:
        esc, alfa = 1.0, (DUR - t_rel) / 0.25
    else:
        esc, alfa = 1.0, 1.0
    pegar(frame, png, 540, 820, esc, alfa)


In [ ]:
%%writefile clipforge/effects.py
"""Dirección de efectos por clip (determinista por semilla):

- zoom base lento (in/out alternado según el clip)
- punch zooms en picos de excitación acústica y momentos con emoji
- camera shake amortiguado en el pico principal
- flash blanco breve en los 1-2 picos más fuertes
- motion blur ligero durante los punch (mezcla de fotogramas en el renderer)
"""
import numpy as np


class PlanFX:
    def __init__(self, dur, seed=0, activo=True, base_max=1.06, punch_amp=0.10):
        self.dur = dur
        self.activo = activo
        rng = np.random.RandomState(seed)
        self.dir_in = bool(rng.randint(2))
        self.base_max = base_max
        self.punch_amp = punch_amp
        self.punches, self.shakes, self.flashes = [], [], []

    def poblar(self, picos_out, emo_ts):
        """Coloca efectos en picos de audio y momentos de emoji."""
        if not self.activo:
            return self
        cands = sorted({round(t, 2) for t in list(picos_out) + list(emo_ts)
                        if 1.2 < t < self.dur - 1.2})
        sel = []
        for t in cands:
            if all(abs(t - x) >= 5.0 for x in sel):
                sel.append(t)
            if len(sel) >= 4:
                break
        self.punches = sel
        if sel:
            self.shakes = [sel[0]]
            self.flashes = sel[:2]
        return self

    def zoom(self, t):
        if not self.activo:
            return 1.0
        p = t / max(self.dur, 0.1)
        z = 1.0 + (self.base_max - 1.0) * (p if self.dir_in else (1 - p))
        for t0 in self.punches:
            d = t - t0
            if 0 <= d < 0.9:
                env = (d / 0.08) if d < 0.08 else float(np.exp(-(d - 0.08) / 0.30))
                z += self.punch_amp * env
        return min(z, 1.20)

    def dzoom(self, t):
        """Velocidad de zoom (para activar el motion blur)."""
        return abs(self.zoom(t + 0.033) - self.zoom(t)) * 30.0

    def shake(self, t):
        for t0 in self.shakes:
            d = t - t0
            if 0 <= d < 0.45:
                a = 8.0 * (1 - d / 0.45)
                return (a * np.sin(2 * np.pi * 11 * d),
                        0.6 * a * np.sin(2 * np.pi * 9 * d + 1.3))
        return (0.0, 0.0)

    def flash(self, t):
        for t0 in self.flashes:
            d = t - t0
            if 0 <= d < 0.14:
                return 0.6 * (1 - d / 0.14)
        return 0.0


In [ ]:
%%writefile clipforge/renderer.py
"""Render final de cada clip en 3 etapas:

1) Vídeo : lectura EDL del máster → reencuadre dinámico → efectos →
           emojis → barra de progreso → subtítulos ASS → x264 (tubería raw)
2) Audio : cortes EDL → mejora (highpass, de-noise, compresor, EQ,
           loudnorm -14 LUFS, limitador) → música opcional con ducking
3) Mux   : vídeo + audio + faststart
"""
import subprocess as sp
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

from . import emojis as emo_mod
from .utils import run, hex_rgb

SALIDA_W, SALIDA_H = 1080, 1920


def _interp_ruta(ruta, t):
    return (float(np.interp(t, ruta["t"], ruta["cx"])),
            float(np.interp(t, ruta["t"], ruta["cy"])))


def render_video(src, job, cfg, tmp_video, fuente_dir="fonts"):
    tm, ruta, fx = job["tm"], job["ruta"], job["fx"]
    fps = cfg["fps"]
    n_frames = max(1, int(tm.out_dur * fps))
    W, H = job["W"], job["H"]
    base_cw = min(W, int(H * 9 / 16))
    pngs = {p: cv2.imread(p, cv2.IMREAD_UNCHANGED) for _, p in job["emos"]}
    color_barra = np.array(hex_rgb(cfg["color_resaltado"])[::-1], np.uint8)  # BGR
    vf = f"ass={Path(job['ass']).name}:fontsdir={fuente_dir}"
    cmd = ["ffmpeg", "-y", "-loglevel", "error",
           "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{SALIDA_W}x{SALIDA_H}",
           "-r", str(fps), "-i", "pipe:",
           "-vf", vf, "-c:v", "libx264", "-preset", "veryfast",
           "-crf", str(cfg["crf"]), "-pix_fmt", "yuv420p", tmp_video]
    pr = sp.Popen(cmd, stdin=sp.PIPE, stderr=sp.PIPE, cwd=str(Path(job["ass"]).parent))
    cap = cv2.VideoCapture(src)
    cur_t, frame_src, prev_out = -1.0, None, None
    try:
        for n in tqdm(range(n_frames), desc=f"   🎬 clip_{job['idx']:03d}",
                      unit="fr", leave=False):
            t_out = n / fps
            t_src = tm.out2src(t_out)
            if t_src < cur_t - 0.05 or t_src > cur_t + 1.5:     # salto de rango EDL
                cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, t_src * 1000.0 - 40))
                cur_t = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
                frame_src = None
            intentos = 0
            while (frame_src is None or cur_t < t_src - 0.001) and intentos < 90:
                pos = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
                ok, fr = cap.read()
                if not ok:
                    break
                frame_src, cur_t = fr, pos
                intentos += 1
            if frame_src is None:
                frame_out = np.zeros((SALIDA_H, SALIDA_W, 3), np.uint8)
            else:
                z = fx.zoom(t_out)
                dx, dy = fx.shake(t_out)
                cw, ch = base_cw / z, H / z
                cx, cy = _interp_ruta(ruta, t_out)
                x0 = int(np.clip(cx - cw / 2 + dx, 0, W - cw))
                y0 = int(np.clip(cy - ch / 2 + dy, 0, H - ch))
                rec = frame_src[y0:y0 + int(ch), x0:x0 + int(cw)]
                interp = cv2.INTER_AREA if cw > SALIDA_W else cv2.INTER_LINEAR
                frame_out = cv2.resize(rec, (SALIDA_W, SALIDA_H), interpolation=interp)
                if cfg.get("motion_blur") and prev_out is not None and fx.dzoom(t_out) > 0.9:
                    frame_out = cv2.addWeighted(frame_out, 0.65, prev_out, 0.35, 0)
                fa = fx.flash(t_out)
                if fa > 0:
                    frame_out = cv2.addWeighted(frame_out, 1 - fa,
                                                np.full_like(frame_out, 255), fa, 0)
            prev_out = frame_out.copy() if cfg.get("motion_blur") else None
            for t0, p in job["emos"]:
                png = pngs.get(p)
                if png is not None:
                    emo_mod.dibujar(frame_out, png, t_out - t0)
            if cfg.get("barra"):
                ancho = int(SALIDA_W * min(1.0, t_out / max(tm.out_dur, 0.1)))
                frame_out[SALIDA_H - 12:, :ancho] = color_barra
            try:
                pr.stdin.write(frame_out.tobytes())
            except (BrokenPipeError, OSError):
                break
    finally:
        cap.release()
        if pr.stdin:
            pr.stdin.close()
        err = pr.stderr.read().decode(errors="ignore") if pr.stderr else ""
        if pr.wait() != 0:
            raise RuntimeError("ffmpeg (vídeo) falló:\n" + err[-1200:])
    return tmp_video


def render_audio(src, job, cfg, tmp_audio, ruta_musica=None):
    ini, fin = job["clip"]["start"], job["clip"]["end"]
    tm = job["tm"]
    wav_src = tmp_audio.replace(".wav", "_src.wav")
    run(["ffmpeg", "-y", "-loglevel", "error", "-ss", f"{ini:.3f}",
         "-t", f"{fin - ini:.3f}", "-i", src, "-vn", "-ac", "2", "-ar", "48000", wav_src])
    n = len(tm.edl)
    if n == 1:
        s, e = tm.edl[0]
        fc = f"[0:a]atrim={s - ini:.3f}:{e - ini:.3f},asetpts=PTS-STARTPTS[cat];"
    else:
        splits = "".join(f"[i{i}]" for i in range(n))
        partes = [f"[0:a]asplit={n}{splits}"]
        for i, (s, e) in enumerate(tm.edl):
            partes.append(f"[i{i}]atrim={s - ini:.3f}:{e - ini:.3f},asetpts=PTS-STARTPTS[s{i}]")
        fc = (";".join(partes) + ";" + "".join(f"[s{i}]" for i in range(n)) +
              f"concat=n={n}:v=0:a=1[cat];")
    tempo = f"atempo={tm.v:.3f}," if abs(tm.v - 1) > 1e-3 else ""
    fc += ("[cat]highpass=f=70,afftdn=nf=-25,"
           "acompressor=threshold=-18dB:ratio=3:attack=6:release=180,"
           "equalizer=f=3000:t=q:w=1:g=1.5," + tempo +
           "loudnorm=I=-14:TP=-1.5:LRA=11,alimiter=limit=0.97,aresample=48000[voz]")
    entradas = ["-i", wav_src]
    mapa = "[voz]"
    if ruta_musica:
        d = tm.out_dur
        entradas += ["-stream_loop", "-1", "-i", ruta_musica]
        fc += (f";[voz]asplit[v1][v2];"
               f"[1:a]atrim=0:{d:.2f},asetpts=PTS-STARTPTS,volume={cfg['vol_musica']:.2f},"
               f"afade=t=in:d=1.2,afade=t=out:st={max(0, d - 2):.2f}:d=2[mus];"
               f"[mus][v1]sidechaincompress=threshold=0.02:ratio=12:attack=5:release=400[duck];"
               f"[v2][duck]amix=inputs=2:duration=first:weights=3 1[mezcla]")
        mapa = "[mezcla]"
    run(["ffmpeg", "-y", "-loglevel", "error"] + entradas +
        ["-filter_complex", fc, "-map", mapa, "-ar", "48000", "-ac", "2", tmp_audio])
    return tmp_audio


def render_clip(src, job, cfg, dir_tmp, fuente_dir="fonts", ruta_musica=None):
    dir_tmp = Path(dir_tmp)
    base = f"clip_{job['idx']:03d}"
    tv = str(dir_tmp / f"{base}_v.mp4")
    ta = str(dir_tmp / f"{base}_a.wav")
    out = str(dir_tmp / f"{base}.mp4")
    render_video(src, job, cfg, tv, fuente_dir)
    render_audio(src, job, cfg, ta, ruta_musica)
    run(["ffmpeg", "-y", "-loglevel", "error", "-i", tv, "-i", ta,
         "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest",
         "-movflags", "+faststart", out])
    return out


In [ ]:
%%writefile clipforge/exporter.py
"""Exportación a Drive: clip_NNN.mp4 + clip_NNN.json + RESUMEN.txt."""
import json
import shutil
from pathlib import Path

from .utils import t2str


def exportar_clip(ruta_local, job, dir_salida):
    dir_salida = Path(dir_salida)
    dir_salida.mkdir(parents=True, exist_ok=True)
    c = job["clip"]
    nombre = f"clip_{job['idx']:03d}"
    destino = dir_salida / f"{nombre}.mp4"
    shutil.copyfile(ruta_local, destino)
    meta = {
        "archivo": destino.name,
        "inicio": c["start"], "fin": c["end"],
        "inicio_hms": t2str(c["start"]), "fin_hms": t2str(c["end"]),
        "duracion_original": round(c["end"] - c["start"], 2),
        "duracion_clip": round(job["tm"].out_dur, 2),
        "score_viral": c.get("score", 0),
        "score_desglose": {"heuristico": c.get("score_heur", c.get("score")),
                           "llm": c.get("score_llm")},
        "tema": c.get("tema", ""),
        "resumen": c.get("resumen", ""),
        "keywords": c.get("keywords", []),
        "titulo": c.get("titulo", ""),
        "hook": c.get("hook", ""),
        "hashtags": c.get("hashtags", []),
        "emojis_texto": [m.get("emoji", "") for m in c.get("emojis", []) if isinstance(m, dict)],
        "n_emojis_en_video": len(job["emos"]),
        "caras_detectadas_pct": round(job["ruta"]["ratio"] * 100),
        "seleccion": c.get("fuente", ""),
    }
    (dir_salida / f"{nombre}.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return destino, meta


def resumen_global(resultados, dir_salida, video):
    lineas = [f"CLIPS GENERADOS · {video}", "=" * 64, ""]
    for destino, meta in resultados:
        lineas += [f"▶ {meta['archivo']}  ·  {meta['inicio_hms']} → {meta['fin_hms']}"
                   f"  ·  ⭐ {meta['score_viral']}",
                   f"  Título:   {meta['titulo']}",
                   f"  Tema:     {meta['tema']}",
                   f"  Hashtags: {' '.join(meta['hashtags'])}", ""]
    p = Path(dir_salida) / "RESUMEN.txt"
    p.write_text("\n".join(lineas), encoding="utf-8")
    return p


In [ ]:
%%writefile clipforge/diarization.py
"""[ROADMAP] Diarización de hablantes con pyannote.audio.

Interfaz preparada: con un HF_TOKEN (huggingface.co) se podrá etiquetar
cada palabra con su hablante para (a) reencuadrar a quien habla en cada
momento —no solo a la cara más grande— y (b) puntuar mejor los debates.
La integración está desacoplada: ningún módulo depende de esta función.
"""


def diarizar(wav, hf_token=None):
    if not hf_token:
        raise NotImplementedError(
            "Configura HF_TOKEN en los Secretos de Colab y descomenta la "
            "integración pyannote para activar la diarización.")
    # from pyannote.audio import Pipeline
    # pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
    #                                 use_auth_token=hf_token)
    # return pipe(wav)


In [ ]:
#@title ⚙️ 2 · Configuración { display-mode: "form" }
import os, json, re, time
from pathlib import Path

#@markdown ### 📹 Entrada / salida (rutas dentro de MyDrive)
CARPETA_ENTRADA = "VideosGP"  #@param {type:"string"}
NOMBRE_VIDEO = "USA_ LA HORA DEL GOLPE SOBRE LA MESA.mp4"  #@param {type:"string"}
CARPETA_SALIDA = "VideosGP/Clips"  #@param {type:"string"}

#@markdown ### ✂️ Clips
NUM_CLIPS = 8  #@param {type:"slider", min:1, max:20, step:1}
DURACION_MINIMA = 18  #@param {type:"slider", min:10, max:40, step:1}
DURACION_MAXIMA = 68  #@param {type:"slider", min:40, max:120, step:1}
IDIOMA = "es"  #@param {type:"string"}
TAMANO_WHISPER = "large-v3"  #@param ["large-v3", "large-v3-turbo", "medium", "small"]

#@markdown ### 🧠 Canal (contexto para el director LLM)
NOMBRE_CANAL = "PecinoGP"  #@param {type:"string"}
CONTEXTO_CANAL = "Canal en español de motociclismo deportivo: MotoGP, WSBK y competición. Directos de análisis, noticias y tertulia con opiniones fuertes."  #@param {type:"string"}

#@markdown ### 🎬 Funciones (activar / desactivar)
SUBTITULOS = True  #@param {type:"boolean"}
EMOJIS = True  #@param {type:"boolean"}
EFECTOS_ZOOM = True  #@param {type:"boolean"}
RECORTE_INTELIGENTE = True  #@param {type:"boolean"}
QUITAR_SILENCIOS = True  #@param {type:"boolean"}
MUSICA = False  #@param {type:"boolean"}
RUTA_MUSICA = "VideosGP/musica.mp3"  #@param {type:"string"}

#@markdown ### 🎨 Estilo
FUENTE = "Poppins ExtraBold"  #@param ["Poppins ExtraBold", "Anton", "Archivo Black"]
COLOR_RESALTADO = "#FFE600"  #@param {type:"string"}
MAYUSCULAS = True  #@param {type:"boolean"}
ROTULO_HOOK = True  #@param {type:"boolean"}
MARCA_DE_AGUA = "@PecinoGP"  #@param {type:"string"}
BARRA_PROGRESO = True  #@param {type:"boolean"}

#@markdown ### 🛠️ Avanzado
FPS = 30  #@param [30, 60] {type:"raw"}
CALIDAD_CRF = 19  #@param {type:"slider", min:16, max:26, step:1}
VELOCIDAD_GLOBAL = 1.0  #@param {type:"slider", min:1.0, max:1.15, step:0.01}
MAX_PAUSA = 0.7  #@param {type:"slider", min:0.3, max:1.5, step:0.1}
FORZAR_TRANSCRIPCION = False  #@param {type:"boolean"}
FORZAR_ANALISIS = False  #@param {type:"boolean"}
FORZAR_SELECCION = False  #@param {type:"boolean"}

CFG = dict(
    num_clips=NUM_CLIPS, duraciones=(20, 30, 45, 60),
    dur_min=DURACION_MINIMA, dur_max=DURACION_MAXIMA,
    idioma=IDIOMA, whisper=TAMANO_WHISPER,
    canal=NOMBRE_CANAL, contexto_canal=CONTEXTO_CANAL,
    subtitulos=SUBTITULOS, mayusculas=MAYUSCULAS, fuente=FUENTE,
    color_resaltado=COLOR_RESALTADO,
    emojis=EMOJIS, max_emojis=6,
    efectos=EFECTOS_ZOOM, motion_blur=True, velocidad=VELOCIDAD_GLOBAL,
    reencuadre=("auto" if RECORTE_INTELIGENTE else "centro"),
    quitar_silencios=QUITAR_SILENCIOS, max_pausa=MAX_PAUSA,
    musica=MUSICA, ruta_musica=RUTA_MUSICA, vol_musica=0.16,
    rotulo=ROTULO_HOOK, marca_agua=MARCA_DE_AGUA, barra=BARRA_PROGRESO,
    fps=FPS, crf=CALIDAD_CRF,
    forzar_transcripcion=FORZAR_TRANSCRIPCION,
    forzar_analisis=FORZAR_ANALISIS, forzar_seleccion=FORZAR_SELECCION,
)
assert CFG["dur_min"] < CFG["dur_max"], "La duración mínima debe ser menor que la máxima"

WORK = Path("/content/clipwork")
_ARCHIVOS_FUENTE = {"Poppins ExtraBold": "Poppins-ExtraBold.ttf",
                    "Anton": "Anton-Regular.ttf",
                    "Archivo Black": "ArchivoBlack-Regular.ttf"}
FUENTE_ASS = FUENTE if (WORK / "fonts" / _ARCHIVOS_FUENTE[FUENTE]).exists() else "DejaVu Sans"
if FUENTE_ASS == "DejaVu Sans":
    print("⚠️ Fuente no descargada → usando DejaVu Sans como respaldo")
print(f"✅ Config: {NUM_CLIPS} clips · {DURACION_MINIMA}-{DURACION_MAXIMA}s · Whisper {TAMANO_WHISPER}")
print(f"   subtítulos={SUBTITULOS} · emojis={EMOJIS} · efectos={EFECTOS_ZOOM} · "
      f"reencuadre={CFG['reencuadre']} · silencios={QUITAR_SILENCIOS} · música={MUSICA}")


In [ ]:
#@title 📂 3 · Montar Google Drive
from google.colab import drive
drive.mount("/content/drive")

import unicodedata
from clipforge.utils import Paso, run, t2str, ffprobe_info

DRIVE_ROOT = "/content/drive/MyDrive"


def _norm(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "", s)


VIDEO_PATH = Path(DRIVE_ROOT) / CARPETA_ENTRADA / NOMBRE_VIDEO
if not VIDEO_PATH.exists():
    print("El nombre no coincide exactamente; buscando en la carpeta…")
    objetivo = _norm(NOMBRE_VIDEO)
    carpeta = Path(DRIVE_ROOT) / CARPETA_ENTRADA
    cands = list(carpeta.glob("*.mp4")) if carpeta.exists() else []
    filtro = [p for p in cands if objetivo[:18] in _norm(p.name) or _norm(p.name) in objetivo]
    cands = filtro or cands
    if not cands:
        raise FileNotFoundError(
            f"No encuentro '{NOMBRE_VIDEO}' en MyDrive/{CARPETA_ENTRADA}. Revisa la sección 2.")
    VIDEO_PATH = cands[0]

OUT_DIR = Path(DRIVE_ROOT) / CARPETA_SALIDA
CACHE_DIR = OUT_DIR / "cache" / re.sub(r"[^\w\s-]", "", VIDEO_PATH.stem).strip()[:60]
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"🎬 Vídeo:  {VIDEO_PATH.name}")
print(f"📁 Salida: {OUT_DIR}")
print(f"💾 Caché:  {CACHE_DIR}")


In [ ]:
#@title 🎞️ 4 · Cargar vídeo (copia local + análisis técnico)
INFO = ffprobe_info(str(VIDEO_PATH))
W, H, SRC_FPS, DURACION, TAMANO = INFO["w"], INFO["h"], INFO["fps"], INFO["dur"], INFO["bytes"]
print(f"📺 {W}x{H} · {SRC_FPS:.0f} fps · {t2str(DURACION)} · {TAMANO / 1e9:.2f} GB")

SRC = "/content/source_video.mp4"
if os.path.exists(SRC) and os.path.getsize(SRC) == TAMANO:
    print("💾 Copia local ya existente, no se vuelve a copiar")
else:
    with Paso("Copiando el vídeo al disco local de Colab (acelera todo el pipeline)"):
        with open(VIDEO_PATH, "rb") as fi, open(SRC, "wb") as fo:
            copiado = 0
            while True:
                b = fi.read(64 * 1024 * 1024)
                if not b:
                    break
                fo.write(b)
                copiado += len(b)
                print(f"\r   {copiado / TAMANO * 100:5.1f}% ({copiado / 1e9:.2f} GB)", end="")
        print()


In [ ]:
#@title 🎙️ 5 · Transcripción (Whisper · palabra a palabra · con caché)
from tqdm.auto import tqdm
import clipforge.transcriber as trans

AUDIO16 = "/content/audio16k.wav"
_marca = "/content/audio16k.size"
if not (os.path.exists(AUDIO16) and os.path.exists(_marca)
        and open(_marca).read() == str(TAMANO)):
    with Paso("Extrayendo audio 16 kHz mono"):
        run(["ffmpeg", "-y", "-loglevel", "error", "-i", SRC,
             "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", AUDIO16])
        open(_marca, "w").write(str(TAMANO))

_bar = tqdm(total=100.0, desc="   Transcribiendo", unit="%")


def _prog(t, total):
    _bar.n = min(100.0, round(t / max(total, 1) * 100, 1))
    _bar.refresh()


with Paso("Transcripción"):
    PALABRAS, INFO_TRANS = trans.transcribir(
        AUDIO16, modelo=CFG["whisper"], idioma=CFG["idioma"],
        cache=CACHE_DIR / f"transcripcion_{CFG['whisper']}.json",
        forzar=CFG["forzar_transcripcion"], clave_video=TAMANO, progreso=_prog)
_bar.n = 100.0
_bar.close()

if not PALABRAS:
    raise RuntimeError("La transcripción no detectó voz. ¿Es el vídeo correcto?")
FRASES = trans.construir_frases(PALABRAS)
print(f"   🗣️ {len(PALABRAS)} palabras · {len(FRASES)} frases · "
      f"idioma: {INFO_TRANS['idioma']} · audio analizado: {t2str(INFO_TRANS['dur'])}")


In [ ]:
#@title 📊 6 · Análisis IA (acústica + embeddings + semántica → score por frase)
import hashlib
import numpy as np
import pandas as pd
import clipforge.audio_features as afx
import clipforge.semantics as sem
import clipforge.scoring as sco

with Paso("Rasgos acústicos (energía, brillo, gritos, picos)"):
    FEAT = afx.calcular(AUDIO16, cache=CACHE_DIR / "audio_feat.npz",
                        forzar=CFG["forzar_analisis"])
PICOS = afx.picos(FEAT)
print(f"   ⚡ {len(PICOS)} picos de excitación en todo el directo")

TEXTOS = [f["texto"] for f in FRASES]
clave_sem = hashlib.md5("\n".join(TEXTOS).encode()).hexdigest()
sem_npz = CACHE_DIR / "semantica.npz"
VIRAL = NOV = None
if sem_npz.exists() and not CFG["forzar_analisis"]:
    try:
        d = np.load(sem_npz)
        if str(d["clave"]) == clave_sem:
            VIRAL, NOV = d["viral"], d["nov"]
            print(f"   💾 Semántica cargada de caché ({len(VIRAL)} frases)")
    except Exception:
        pass
if VIRAL is None:
    with Paso("Embeddings multilingües + similitud viral + novedad de tema"):
        EMB = sem.calcular(TEXTOS)
        VIRAL, NOV = sem.viralidad(EMB), sem.novedad(EMB)
        np.savez_compressed(sem_npz, clave=np.array(clave_sem),
                            viral=VIRAL, nov=NOV, emb=EMB.astype(np.float16))

with Paso("Fusión de señales por frase"):
    FRASES = sco.puntuar_frases(FRASES, FEAT, VIRAL, NOV)

top = sorted(FRASES, key=lambda f: -f["score"])[:8]
print("\n🔝 Frases con más potencial:")
display(pd.DataFrame([{"min": t2str(f["s"]), "score": f["score"],
                       "frase": f["texto"][:90]} for f in top]))


In [ ]:
#@title 🎯 7 · Detección de clips (candidatos + director LLM + score viral)
import pandas as pd
import clipforge.scoring as sco
import clipforge.llm_director as dire

PLAN_JSON = CACHE_DIR / "plan_clips.json"
PLAN = None
if PLAN_JSON.exists() and not CFG["forzar_seleccion"]:
    try:
        PLAN = json.loads(PLAN_JSON.read_text(encoding="utf-8"))["clips"]
        print(f"💾 Plan cargado de caché ({len(PLAN)} clips).")
        print("   Puedes editar el JSON en Drive y re-ejecutar desde aquí; "
              "marca FORZAR_SELECCION en la sección 2 para regenerarlo.")
    except Exception:
        PLAN = None

if PLAN is None:
    with Paso("Generando ventanas candidatas (20/30/45/60 s)"):
        CANDS = sco.candidatos(FRASES, num=CFG["num_clips"] * 2,
                               duraciones=CFG["duraciones"],
                               dur_min=CFG["dur_min"], dur_max=CFG["dur_max"])
        if len(CANDS) < CFG["num_clips"]:
            CANDS = sco.candidatos(FRASES, num=CFG["num_clips"] * 2,
                                   duraciones=CFG["duraciones"],
                                   dur_min=CFG["dur_min"], dur_max=CFG["dur_max"],
                                   umbral=0.0)
    print(f"   🔍 {len(CANDS)} candidatos")
    with Paso("Director editorial (LLM o heurístico)"):
        PLAN = dire.dirigir(CANDS, FRASES, num=CFG["num_clips"],
                            contexto=CFG["contexto_canal"], canal=CFG["canal"])
    PLAN_JSON.write_text(json.dumps({"video": VIDEO_PATH.name, "clips": PLAN},
                                    ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n🎯 {len(PLAN)} clips seleccionados:")
display(pd.DataFrame([{"clip": c["id"], "inicio": t2str(c["start"]), "fin": t2str(c["end"]),
                       "dur (s)": round(c["end"] - c["start"]), "⭐ score": c["score"],
                       "título": c["titulo"][:60], "fuente": c.get("fuente", "")}
                      for c in PLAN]))


In [ ]:
#@title 🎥 8 · Reencuadre inteligente + eliminación de silencios
import clipforge.cuts as cuts
import clipforge.reframe as ref

TRABAJOS = []
for c in PLAN:
    edl = cuts.construir_edl(PALABRAS, c["start"], c["end"],
                             quitar=CFG["quitar_silencios"], max_pausa=CFG["max_pausa"])
    tm = cuts.TimeMap(edl, velocidad=CFG["velocidad"])
    TRABAJOS.append({"idx": c["id"], "clip": c, "edl": edl, "tm": tm, "W": W, "H": H})

with Paso(f"Seguimiento facial en {len(TRABAJOS)} clips"):
    for job in TRABAJOS:
        job["ruta"] = ref.calcular_ruta(SRC, job["tm"], W, H, modo=CFG["reencuadre"])
        c = job["clip"]
        recortado = (c["end"] - c["start"]) - job["tm"].out_dur * CFG["velocidad"]
        print(f"   clip_{job['idx']:03d} · caras en {job['ruta']['ratio'] * 100:3.0f}% de muestras"
              f" · silencios recortados: {recortado:4.1f}s · duración final {job['tm'].out_dur:.1f}s")


In [ ]:
#@title ✍️ 9 · Subtítulos animados + emojis contextuales
import clipforge.subtitles as subs
import clipforge.emojis as emo

with Paso("Generando subtítulos ASS karaoke y emojis"):
    for job in TRABAJOS:
        c = job["clip"]
        dentro = [w for w in PALABRAS
                  if w["s"] >= c["start"] - 0.05 and w["e"] <= c["end"] + 0.3]
        job["palabras_out"] = job["tm"].remap_palabras(dentro)
        job["ass"] = str(WORK / f"clip_{job['idx']:03d}.ass")
        subs.generar_ass(job["palabras_out"], Path(job["ass"]), dur=job["tm"].out_dur,
                         fuente=FUENTE_ASS, color=CFG["color_resaltado"],
                         mayusculas=CFG["mayusculas"],
                         hook=(c.get("hook") or "") if CFG["rotulo"] else "",
                         marca=CFG["marca_agua"], con_subs=CFG["subtitulos"])
        job["emos"] = emo.momentos(c, job["palabras_out"], WORK / "emoji",
                                   activo=CFG["emojis"], maximo=CFG["max_emojis"])
        print(f"   clip_{job['idx']:03d} · {len(job['palabras_out'])} palabras subtituladas"
              f" · {len(job['emos'])} emojis")


In [ ]:
#@title ⚡ 10 · Efectos dinámicos (zoom, punch, shake, flash)
import clipforge.effects as fxm

for job in TRABAJOS:
    c = job["clip"]
    picos_out = [job["tm"].src2out(p) for p in PICOS if c["start"] < p < c["end"]]
    fx = fxm.PlanFX(job["tm"].out_dur, seed=job["idx"], activo=CFG["efectos"])
    job["fx"] = fx.poblar(picos_out, [t for t, _ in job["emos"]])
    print(f"   clip_{job['idx']:03d} · zoom {'in' if fx.dir_in else 'out'}"
          f" · {len(fx.punches)} punch · {len(fx.shakes)} shake · {len(fx.flashes)} flash")


In [ ]:
#@title 📤 11 · Exportación (render final → Drive)
import clipforge.renderer as rend
import clipforge.exporter as exp

ruta_musica = None
if CFG["musica"]:
    p = Path(DRIVE_ROOT) / CFG["ruta_musica"]
    if p.exists():
        ruta_musica = str(p)
        print(f"🎵 Música de fondo: {p.name} (vol {CFG['vol_musica']}, con ducking)")
    else:
        print(f"⚠️ No existe {p} → los clips saldrán sin música")

RESULTADOS = []
t00 = time.time()
for job in TRABAJOS:
    c = job["clip"]
    print(f"\n🎬 Clip {job['idx']:03d}/{len(TRABAJOS):03d} · ⭐{c['score']} · {c['titulo']}")
    print(f"   {t2str(c['start'])} → {t2str(c['end'])} · {job['tm'].out_dur:.0f}s finales")
    try:
        t0 = time.time()
        local = rend.render_clip(SRC, job, CFG, WORK / "out", "fonts", ruta_musica)
        destino, meta = exp.exportar_clip(local, job, OUT_DIR)
        print(f"   ✅ {destino.name} · {os.path.getsize(local) / 1e6:.1f} MB · "
              f"{time.time() - t0:.0f}s")
        RESULTADOS.append((destino, meta))
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:600]}")

if RESULTADOS:
    exp.resumen_global(RESULTADOS, OUT_DIR, VIDEO_PATH.name)
print(f"\n🏁 {len(RESULTADOS)}/{len(TRABAJOS)} clips en {(time.time() - t00) / 60:.1f} min")
print(f"📁 {OUT_DIR}")


In [ ]:
#@title 🏁 12 · Resumen final + previsualización
import pandas as pd
from base64 import b64encode
from IPython.display import HTML, display

if RESULTADOS:
    display(pd.DataFrame([{"archivo": m["archivo"],
                           "momento": f"{m['inicio_hms']} → {m['fin_hms']}",
                           "dur (s)": m["duracion_clip"], "⭐": m["score_viral"],
                           "tema": m["tema"], "título": m["titulo"][:55]}
                          for _, m in RESULTADOS]))
    print(f"📁 Clips en Drive:   {OUT_DIR}")
    print("📝 Títulos/hashtags: RESUMEN.txt · metadatos: clip_NNN.json")
    print(f"🔁 Para re-editar: modifica {CACHE_DIR / 'plan_clips.json'} y "
          "re-ejecuta desde la sección 7.")
    ruta = WORK / "out" / RESULTADOS[0][1]["archivo"]
    if ruta.exists() and ruta.stat().st_size < 60e6:
        print("\n👀 Previsualización del primer clip:")
        display(HTML('<video controls width="300" src="data:video/mp4;base64,'
                     + b64encode(ruta.read_bytes()).decode() + '"></video>'))
else:
    print("No hay clips generados; revisa los errores de la sección 11.")


## 🛟 Solución de problemas y siguientes pasos

**Problemas típicos**

| Síntoma | Solución |
|---|---|
| `SIN GPU` en la sección 1 | `Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU` y ejecuta todo de nuevo |
| Error cuDNN / CTranslate2 | Reinicia el entorno y ejecuta las celdas en orden (la sección 1 pre-carga las librerías CUDA) |
| La transcripción tarda mucho | Cambia `TAMANO_WHISPER` a `large-v3-turbo` (≈4× más rápido, calidad casi idéntica) |
| Los títulos son mediocres | Añade `GEMINI_API_KEY` en los Secretos 🔑 (gratis en aistudio.google.com/app/apikey) |
| No aparece la cara bien encuadrada | Prueba a desactivar `RECORTE_INTELIGENTE` (usa encuadre centrado) |
| Quiero otros momentos | Edita `Clips/cache/<vídeo>/plan_clips.json` (tiempos, títulos…) y re-ejecuta desde la sección 7 |

**Flujo de re-ejecución (todo cachea en Drive)**

- Cambiar estilo de subtítulos/efectos → re-ejecuta desde la **sección 8** (segundos).
- Cambiar nº de clips o duraciones → marca `FORZAR_SELECCION` y re-ejecuta desde la **7**.
- Otro vídeo → cambia `NOMBRE_VIDEO` en la **2** y ejecuta todo (la transcripción del anterior queda guardada).

**Roadmap preparado en la arquitectura**

- 🎙️ **Diarización** (`clipforge/diarization.py`): reencuadre por hablante activo con pyannote + HF_TOKEN.
- ⏩ **Speed-ramps variables**: el `TimeMap` ya soporta velocidad global (`VELOCIDAD_GLOBAL`); extenderlo a tramos es directo.
- 🎬 **B-roll y zooms por escena**: los picos de `audio_features` y los embeddings dan los puntos de inserción.
- 📱 Subida automática a YouTube/TikTok vía API con los metadatos de `clip_NNN.json`.

---
*ClipForge AI · hecho para PecinoGP · pipeline 100% tuyo, sin límites ni marcas de agua de terceros* 🏍️💨
